### 1.1.1. Set Declaration vs. Definition 
#### Set Declaration and Definition Syntax

In [ ]:
set Name [dimen n] [within SEXPR] [= SEXPR | default SEXPR] ;

A **symbolic definition** uses `=` to bind the set permanently to an expression evaluated at model-generation time. A **data assignment** uses `:=` in a data file or `let` command to supply explicit member values at runtime.

In [ ]:
# Form 1: declaration only — members must come from a data file or amplpy
set PROD;

# Form 2: symbolic definition — fixed to this expression; data cannot override it
set SHIFTS = {"Day", "Evening", "Night"};

# Form 3: default — uses the listed members unless data provides other values
set PROD default {"bands", "coils", "plate"};

#### The `=` Operator (Symbolic Definition)

When a set declaration includes `= EXPR`,

```python
set SHIFTS = {"Day", "Evening", "Night"};
```

 the set is **symbolically defined**: its membership is the result of evaluating `EXPR` during model generation. AMPL re-evaluates the expression whenever any set or parameter it depends on changes. Because the value is fixed by the model, a data section cannot re-assign it.

Symbolic definition is appropriate when:
- The set is a structural invariant of the model (e.g., shift categories, a range of periods).
- The set is derived from other sets or parameters and must track them automatically.
- The set is small and its members are unlikely to change across problem instances.

The expression on the right of `=` may be any valid set expression: a literal set, an arithmetic range, a filtered set comprehension, or a set-algebraic combination.

In [ ]:
# Literal set — fixed members embedded in the model
set SHIFTS = {"Day", "Evening", "Night"};

# Arithmetic range parameterized by data — recomputed if start/end/interval change
param start integer;
param end > start integer;
param interval > 0 integer;
set YEARS = start .. end by interval;

# Filtered comprehension — CHEAP depends on PROD and cost; updated automatically
set PROD;
param cost {PROD} >= 0;
set CHEAP = {p in PROD: cost[p] < 10};

# set-algebraic combination
set PROD_A = {"bands", "coils"};
set PROD_B = {"coils", "plate", "rods"};

set ALL_PROD   = PROD_A union PROD_B;          # {"bands", "coils", "plate", "rods"}
set COMMON     = PROD_A inter PROD_B;          # {"coils"}
set ONLY_A     = PROD_A diff PROD_B;           # {"bands"}
set EXCLUSIVE  = PROD_A symdiff PROD_B;        # {"bands", "plate", "rods"}
set PAIRS      = PROD_A cross PROD_B;          # all ordered pairs (a, b)

#### The `:=` Operator (Data Assignment)

`:=` assigns explicit member values to a set that was declared without `=`. It appears in:
- A **data file** (inside a `data;` section), to load instance data.
- A **`let` command** in an AMPL script or interactive session, to override the current value at runtime.

Unlike the `=` phrase in a declaration, `:=` does not bind the set permanently. A later `let` or data reload can replace the assignment. Using `:=` inside a model declaration is formally deprecated; use `=` or `default` there instead.

The two fragments below show the same set populated first via a data file, then overridden with `let`.

In [ ]:
# --- model file ---
set PROD;

# --- data file (.dat) ---
data;
set PROD := bands coils plate;

# --- interactive session or script
let PROD := {"bands", "coils", "plate", "rods"};  # Replace the set completely
let PROD := PROD union {"rods"};    # Add new members to existing set
let PROD := PROD diff {"bands"};    # Remove members


<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Using <code>:=</code> inside a model-file declaration (e.g., <code>set PROD := {"a", "b"};</code>) is accepted by current AMPL for historical compatibility but is deprecated. The intended meaning of <code>:=</code> in a declaration is identical to <code>=</code>. Write <code>=</code> in the model and reserve <code>:=</code> for data files and <code>let</code> commands.</div>

#### When to Define in the Model vs. a Data File

The choice between embedding set content in the model (`=`) versus supplying it via a data file (`:=`) turns on whether the set is **structural** or **instance-varying**.

| Criterion | Use `=` in model | Use `:=` in data file |
|---|---|---|
| Set is small (a handful of elements) | Yes | Possible but unnecessary |
| Set is a fundamental model invariant | Yes | No — data can inadvertently alter it |
| Set varies across problem instances | No | Yes |
| Set has hundreds or thousands of members | No | Yes |
| Model is reused with different datasets | No | Yes — swap data file, not model |

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
The canonical AMPL discipline is: declare sets in the model, supply their data from outside. This keeps the model file stable and the data file the only thing that changes between runs.</div>

In [ ]:
# Model file — structural, fixed: defined with =
set SHIFTS = {"Day", "Evening", "Night"};  # invariant across all instances

# Model file — instance-varying: declared only
set PROD;    # membership comes from the data file
set CITIES;  # membership comes from the data file

# Data file — instance-specific assignments
data;
set PROD   := bands coils plate rods wire;
set CITIES := GARY CLEV PITT;

#### Set Data Assignment in amplpy:

| # | Pattern | Syntax | Notes |
|---|---------|--------|-------|
| 1 | Raw AMPL | `ampl.eval("set PROD := {'bands', 'coils', 'plate'};")` | Must include `:=` and proper AMPL syntax |
| 2 | Property assignment | `ampl.set["PROD"] = ["bands", "coils", "plate"]` | Set must be declared first |
| 3 | Method assignment | `ampl.get_set("PROD").set_values(["bands", "coils", "plate"])` | Set must be declared first |
| 4 | DataFrame assignment | `ampl.set_data(df, 'PROD')` | Set must be declared first; can include parameters |

In [ ]:
ampl.reset()

# Pattern 1: Raw AMPL - simplest for static data
ampl.eval("set PROD1 := {'bands', 'coils', 'plate'};")

# Pattern 2: Property assignment - most Pythonic
# 🏆 RECOMMENDED: Use this pattern for 80% of cases
ampl.eval("set PROD2;")
ampl.set["PROD2"] = ["bands", "coils", "plate"]

# Pattern 2 variant: From dictionary keys
ampl.eval("set PROD3;")
bounds = {"bands": (0, 100), "coils": (0, 100), "plate": (0, 100)}
ampl.set["PROD3"] = list(bounds.keys())

# Pattern 3: Method assignment - explicit and chainable
ampl.eval("set PROD4;")
ampl.get_set("PROD4").set_values(["bands", "coils", "plate"])

# Pattern 4: DataFrame with tolist() - good for pandas integration
# 🏆 Use this pattern when data comes from files or has parameters
ampl.eval("set PROD5;")
df = pd.DataFrame(["bands", "coils", "plate"], columns=["PROD5"])
ampl.get_set("PROD5").set_values(df["PROD5"].tolist())

#### Derived (Symbolic) Sets

A **derived set** is defined with `=` using an expression that references other sets or parameters. AMPL evaluates the expression during model generation and re-evaluates it if any dependency changes. A **static set** receives its members once from data and does not track any model-level dependencies.

The distinction matters for model maintenance: changing a parameter's data values automatically updates all derived sets that depend on it, with no manual re-specification required.

Derived sets can express:
- **Filtered subsets**: elements of a set satisfying a parameter condition.
- **Cross-products with filters**: pairs from two sets satisfying a joint condition.
- **Set-algebraic combinations**: unions, intersections, differences of other sets.

In [ ]:
set PROD;                          # static — members come from data
param cost {PROD} >= 0;
param market {PROD} >= 0;

# Derived: recomputed whenever cost data changes
set CHEAP = {p in PROD: cost[p] < 10};

# Derived: ordered pairs of distinct products
set PAIRS = {p in PROD, q in PROD: p <> q};

# Derived: set-algebraic union of two declared sets
set MINREQ;  # nutrients with a minimum requirement
set MAXREQ;  # nutrients with a maximum requirement
set NUTR = MINREQ union MAXREQ;

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
A set defined with <code>=</code> cannot receive a value from a data section or a <code>let</code> command. Attempting to assign data to such a set causes an error. If the set must be overridable, use <code>default</code> instead of <code>=</code>.</div>

#### Default Values

The `default EXPR` attribute in a set declaration specifies a **fallback value**: AMPL uses `EXPR` as the set's membership if no data assignment is provided. Unlike `=`, a `default` value can be overridden by a data file, a `let` command, or an amplpy assignment.

Key behaviours:
- If a data section assigns the set, the `default` expression is never evaluated for that instance.
- If no data is supplied and no `let` overrides the set, the default is evaluated once at model-generation time.
- `=` and `default` are mutually exclusive in a single declaration.
- The default expression may be `{}` (the empty set).

Use `default` when the set has a sensible prototype value that should work out of the box, but must remain replaceable for different problem instances.

In [ ]:
# Declaration with default — used when no data assignment is made
set PROD default {"bands", "coils", "plate"};

# Override the default completely via a data file
data;
set PROD := bands coils plate rods wire;  # default is discarded for this instance

# Or override via let in a script
let PROD := {"rods", "wire", "plate"};    # replaces the active membership

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use <code>default {}</code> (the empty set) when a set is optional — for example, a set of active constraints or overridden parameters. This avoids an "undefined set" error if no data is supplied, while keeping the model correct for the empty case.</div>

#### Edge Cases

**Edge case 1: Attempting to assign data to a set defined with `=`**

A set declared with `=` is permanently bound to its expression. Any attempt to supply its value through a data section or `let` command will produce an error. This is a common mistake when converting a model from one that originally defined the set in the data to one with an embedded definition.

In [ ]:
# Model file — set defined with =
set SHIFTS = {"Day", "Evening", "Night"};

# Data file — ERROR: cannot assign to a set defined with =
# set SHIFTS := "Day" "Night";  # this would cause an AMPL error

# Correct fix: use 'default' if override is needed
set SHIFTS default {"Day", "Evening", "Night"};